# Neural Frame Generation on Kaggle: NeuralFG (GPU T4 x2)

**Goal.** Train and evaluate the NeuralFG frame-interpolation network described in `RESEARCH.md` (section 6): given frames t-1 and t, synthesise the middle frame t-0.5. Only Kaggle-hosted datasets are used, and the whole notebook fits an 8 hour end-to-end budget on two Tesla T4 GPUs.

**Approach.**

1. Discover the attached datasets under `/kaggle/input` with a capped, randomised directory walk, keep clean colour frames, and build consecutive triplets (t-1, GT t-0.5, t) inside each sequence folder.
2. Split by video or sequence with a deterministic hash. If the Vimeo-90K triplet mirror ships the official `tri_testlist.txt`, those triplets form the test set (standard protocol); otherwise a hashed set of whole videos is held out. Vid4 is a zero-shot test only.
3. Decode once into a uint8 triplet cache (256 x 256 crops of all three frames, memory-mapped and shared by both GPU processes).
4. Train with DistributedDataParallel (one process per T4), FP16 autocast for the convolutions and FP32 warping, EMA weights, and a learning-rate schedule driven by a fixed wall-clock budget.
5. Evaluate on both GPUs against the frame-average and frame-copy baselines, analyse errors by motion size, visualise flow and blend weights, measure latency at 540p and 1080p, export ONNX, and zip all outputs.

**Model.** NeuralFG with RGB input (6 channels; the engine build adds motion vectors and two depth maps for 10 channels): average-pool by 4, three 32-channel 3x3 convolutions with LeakyReLU, and a 3x3 head with 9 outputs (two flows, two blend logits, an RGB residual), about 22.9K parameters. Both frames are backward-warped to t-0.5 and blended with softmax-normalised weights. The head starts at zero, so training begins from the frame average. Loss: Charbonnier plus 0.5 x a 7x7 soft ternary census loss (the perceptual VGG term in `RESEARCH.md` needs downloaded weights and is left out).

**Datasets** (attach with **Add Input**; any missing one is skipped and its share redistributed):

| Kaggle dataset | Role |
| :--- | :--- |
| `chenshu123/vimeo-triplet` | Training; official test list used as the test set when present |
| `wangsally/vimeo-90k-7` | Training (sliding triplets from septuplets; official test septuplets excluded when the list is present) |
| `amithkesavmrajagiri/reds-dataset` | Training (720p sequences, larger motion) |
| `artemmmtry/mpi-sintel-dataset` | Training (clean and final passes; flow, occlusion and depth folders ignored) |
| `uom200647r/vid4-dataset` | Zero-shot test only |

FlyingChairs is not used: it provides image pairs without a middle frame, so it cannot supervise interpolation, and the model's flows are not defined against its ground-truth flow.

**Hardware.** Kaggle GPU T4 x2 (2 x 15 GB VRAM, fp16 tensor cores, no fast bf16), 4 CPU cores, about 30 GB RAM, 20 GB of saved disk in `/kaggle/working`.

**Required Kaggle settings.** Accelerator: GPU T4 x2. Internet: not needed. Attach the datasets above. Use **Save Version > Save & Run All** for an unattended run.

**Run budget** (upper bounds enforced in code):

| Stage | Budget |
| :--- | :--- |
| Setup, discovery, EDA | about 10 min |
| Triplet cache build | at most 25 min |
| Training (DDP, both GPUs) | 5 h, cut automatically if earlier stages ran long |
| Evaluation, Vid4, latency, export, zip | about 30 min (55 min reserved) |
| **Total** | **about 6.5 h, hard ceiling 8 h** |

Set `QUICK_RUN = True` in the configuration cell for a smoke test of about 30 minutes.

**Outputs** in `/kaggle/working/`:

```text
code/          generated library and DDP training script
weights/       fg_last.pt, fg_best_ema.pt, fg_model.pt, neural_fg.onnx
plots/         every figure shown in the notebook, as PNG
metrics/       JSON and CSV metrics (hardware, catalogue, training history, test results, latency, summary)
logs/          notebook log and the full DDP training log
predictions/   sample interpolated frames
outputs.zip    everything above, created by the final cell
```

The temporary triplet cache (`cache/`) is deleted before zipping.

## 1. Setup

Imports, output folders, a logger that writes to the screen and to `logs/notebook.log`, a `stage` timer that records the run time of every stage for the final report, and a `show` helper that displays a figure once and saves it to `plots/`. `NOTEBOOK_START` anchors the 8 hour budget.

In [ ]:
import copy
import json
import logging
import math
import os
import random
import shutil
import subprocess
import sys
import threading
import time
import zipfile
from concurrent.futures import ThreadPoolExecutor, as_completed
from contextlib import contextmanager
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil
import torch
from IPython.display import FileLink, display
from matplotlib.colors import hsv_to_rgb
from PIL import Image
from tqdm.auto import tqdm

NOTEBOOK_START = time.time()
INPUT_ROOT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
DIRS = {
    name: WORK / name
    for name in ["code", "cache", "weights", "plots", "metrics", "logs", "predictions"]
}
for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

log = logging.getLogger("neural_fg")
log.setLevel(logging.INFO)
log.handlers.clear()
for handler in (
    logging.StreamHandler(sys.stdout),
    logging.FileHandler(DIRS["logs"] / "notebook.log"),
):
    handler.setFormatter(logging.Formatter("%(asctime)s | %(message)s", "%H:%M:%S"))
    log.addHandler(handler)

STAGE_TIMES = {}


@contextmanager
def stage(name):
    start = time.time()
    log.info("start: %s", name)
    try:
        yield
    finally:
        STAGE_TIMES[name] = time.time() - start
        log.info(
            "done: %s in %.1f min (run total %.2f h)",
            name,
            STAGE_TIMES[name] / 60,
            (time.time() - NOTEBOOK_START) / 3600,
        )


def show(fig, name):
    fig.savefig(DIRS["plots"] / f"{name}.png", dpi=130, bbox_inches="tight")
    display(fig)
    plt.close(fig)


def save_json(obj, name):
    with open(DIRS["metrics"] / name, "w") as f:
        json.dump(obj, f, indent=2, default=str)


log.info("output folders ready under %s", WORK)

## 2. Hardware check

Detects the GPUs, CPU cores, RAM and free disk at runtime, prints them, saves `metrics/hardware.json`, and stops early if two GPUs are not visible, because training launches one process per GPU. Expected output: two Tesla T4 rows with about 15 GB each.

In [ ]:
N_GPUS = torch.cuda.device_count()
gpu_rows = []
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    gpu_rows.append(
        {
            "gpu": i,
            "name": props.name,
            "vram_gb": round(props.total_memory / 2**30, 2),
            "multiprocessors": props.multi_processor_count,
            "compute_capability": f"{props.major}.{props.minor}",
        }
    )
vm, disk = psutil.virtual_memory(), shutil.disk_usage(WORK)
HARDWARE = {
    "gpus": N_GPUS,
    "cpu_count": os.cpu_count(),
    "ram_total_gb": round(vm.total / 2**30, 1),
    "ram_available_gb": round(vm.available / 2**30, 1),
    "disk_free_gb_working": round(disk.free / 2**30, 1),
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_list": gpu_rows,
}
print(pd.DataFrame(gpu_rows).to_string(index=False))
print(pd.Series({k: v for k, v in HARDWARE.items() if k != "gpu_list"}).to_string())
save_json(HARDWARE, "hardware.json")
assert (
    N_GPUS >= 2
), "Two GPUs are required. Choose Settings > Accelerator > GPU T4 x2 and run again."

## 3. Configuration

Every tunable value lives here.

* `cache_patch` 256 / `crop` 224: cached triplets are 256 px (the full height of Vimeo frames) so each batch takes a random 224 px crop; flips, transposes and time reversal are applied on the GPU. 224 is divisible by the 4x pooling.
* `cache_gb` caps the triplet cache (each triplet is 0.59 MB); the build cell lowers it if RAM or disk is short.
* `train_hours` is the wall-clock training budget that drives warm-up plus cosine decay; `eval_reserve_h` keeps time for evaluation inside `RUN_BUDGET_H`.
* `base_lr` applies at an effective batch of `base_batch`, scaled by the square root of the ratio for the probed batch; `max_batch_per_gpu` caps the probe, because past GPU saturation a larger batch only cuts the number of updates in a fixed time.
* Per dataset: `quota` (share of cached triplets, renormalised and redistributed when a dataset is short), `group_level` (1 sequence folder, 2 video folder) and `max_per_folder` (triplets taken from one folder, so long sequences cannot dominate).

In [ ]:
QUICK_RUN = False
RUN_BUDGET_H = 8.0

CFG = {
    "seed": 42,
    "mid_channels": 32,
    "downsample": 4,
    "cache_patch": 256,
    "crop": 224,
    "cache_gb": 9.0,
    "cache_build_min": 25,
    "min_patch_std": 4.0,
    "val_triplets": 1024,
    "val_permille": 30,
    "test_permille": 30,
    "list_cap_files": 60000,
    "eda_samples": 400,
    "test_max": 1500,
    "holdout_full_per_dataset": 200,
    "train_hours": 5.0,
    "eval_reserve_h": 0.9,
    "base_lr": 4e-4,
    "base_batch": 64,
    "max_lr": 1.5e-3,
    "min_lr": 1e-6,
    "warmup_frac": 0.03,
    "weight_decay": 1e-4,
    "w_census": 0.5,
    "ema_decay": 0.999,
    "grad_clip": 1.0,
    "batch_candidates": [16, 32, 64, 96, 128, 192, 256, 384],
    "max_batch_per_gpu": 128,
    "vram_headroom": 0.10,
    "val_interval_min": 10,
    "progress_interval_s": 15,
    "sync_every": 10,
    "val_batch": 32,
    "latency_iters": 100,
    "latency_sizes": [[540, 960], [1080, 1920]],
}
SOURCES = [
    {
        "name": "vimeo_triplet",
        "slug": "chenshu123/vimeo-triplet",
        "role": "train",
        "group_level": 2,
        "quota": 0.45,
        "max_per_folder": 1,
    },
    {
        "name": "vimeo_septuplet",
        "slug": "wangsally/vimeo-90k-7",
        "role": "train",
        "group_level": 2,
        "quota": 0.25,
        "max_per_folder": 5,
    },
    {
        "name": "reds",
        "slug": "amithkesavmrajagiri/reds-dataset",
        "role": "train",
        "group_level": 1,
        "quota": 0.18,
        "max_per_folder": 15,
    },
    {
        "name": "sintel",
        "slug": "artemmmtry/mpi-sintel-dataset",
        "role": "train",
        "group_level": 1,
        "quota": 0.12,
        "max_per_folder": 40,
    },
    {
        "name": "vid4",
        "slug": "uom200647r/vid4-dataset",
        "role": "video_test",
        "group_level": 1,
        "max_per_folder": 1000,
    },
]
if QUICK_RUN:
    CFG.update(
        {
            "cache_gb": 0.6,
            "cache_build_min": 5,
            "val_triplets": 128,
            "list_cap_files": 6000,
            "eda_samples": 100,
            "test_max": 150,
            "holdout_full_per_dataset": 20,
            "train_hours": 10 / 60,
            "eval_reserve_h": 0.5,
            "val_interval_min": 3,
            "latency_iters": 30,
        }
    )

random.seed(CFG["seed"])
np.random.seed(CFG["seed"])
torch.manual_seed(CFG["seed"])
torch.backends.cudnn.benchmark = True
save_json(
    {
        "quick_run": QUICK_RUN,
        "run_budget_h": RUN_BUDGET_H,
        "cfg": CFG,
        "sources": SOURCES,
    },
    "config.json",
)
print(pd.Series(CFG).to_string())

## 4. Library: model, losses, metrics, augmentation

Written to `code/fg_lib.py` so the notebook and the DDP processes import the same code.

* `backward_warp`: bilinear `grid_sample` in FP32 (pixel-centre convention, border padding), as `RESEARCH.md` requires for warp coordinates.
* `NeuralFG`: replicate-pads to a multiple of 4, pools, runs the CNN, upsamples the 9-channel prediction bilinearly, scales flows to full-resolution pixels, warps both frames and blends them with softmax weights plus the residual. `FGInference` wraps it for export.
* Losses: Charbonnier and a 7x7 soft ternary census distance on luma (robust to brightness changes between frames).
* Metrics: PSNR and SSIM on RGB in [0, 1] after 8-bit rounding (the usual Vimeo-90K protocol), and the Middlebury interpolation error (RMS on the 0-255 scale).
* `augment_triplets`: the same flip or transpose for all three frames of a sample, and random time reversal (swap t-1 and t).

In [ ]:
%%writefile /kaggle/working/code/fg_lib.py
"""NeuralFG frame-interpolation library: model, warping, losses, metrics and augmentation."""
import torch
import torch.nn as nn
import torch.nn.functional as F


def backward_warp(img, flow):
    """Sample img at (x + flow_x, y + flow_y) for every output pixel; flow in pixels, FP32."""
    n, _, h, w = img.shape
    ys, xs = torch.meshgrid(torch.arange(h, device=img.device, dtype=torch.float32),
                            torch.arange(w, device=img.device, dtype=torch.float32), indexing="ij")
    gx = (xs[None] + flow[:, 0] + 0.5) * (2.0 / w) - 1.0
    gy = (ys[None] + flow[:, 1] + 0.5) * (2.0 / h) - 1.0
    grid = torch.stack([gx, gy], dim=-1)
    return F.grid_sample(img.float(), grid, mode="bilinear", padding_mode="border", align_corners=False)


class NeuralFG(nn.Module):
    """Quarter-resolution intermediate-flow estimator with occlusion-weighted blending.

    Inputs are frames t-1 and t (RGB here; the engine build adds motion vectors and two depths,
    10 channels). A 4-layer CNN at 1/downsample resolution predicts, per pixel: flow from the
    middle frame to each input (in low-resolution pixels, scaled to full resolution), two blend
    logits normalised with a softmax, and an RGB residual. The last layer starts at zero, so an
    untrained model outputs the plain average of the two frames.
    """

    def __init__(self, in_ch=6, mid=32, downsample=4):
        super().__init__()
        self.downsample = downsample
        self.encoder = nn.Sequential(
            nn.Conv2d(in_ch, mid, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(mid, mid, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(mid, 9, 3, padding=1),
        )
        nn.init.zeros_(self.encoder[-1].weight)
        nn.init.zeros_(self.encoder[-1].bias)

    def forward(self, f0, f1):
        h, w = f0.shape[-2:]
        d = self.downsample
        pad = (0, (-w) % d, 0, (-h) % d)
        if pad[1] or pad[3]:
            f0, f1 = F.pad(f0, pad, mode="replicate"), F.pad(f1, pad, mode="replicate")
        pred = self.encoder(F.avg_pool2d(torch.cat([f0, f1], dim=1), d))
        pred = F.interpolate(pred.float(), size=f0.shape[-2:], mode="bilinear", align_corners=False)
        flow0, flow1 = pred[:, 0:2] * d, pred[:, 2:4] * d
        weights = torch.softmax(pred[:, 4:6], dim=1)
        out = weights[:, 0:1] * backward_warp(f0, flow0) + weights[:, 1:2] * backward_warp(f1, flow1) + pred[:, 6:9]
        aux = {"flow0": flow0[..., :h, :w], "flow1": flow1[..., :h, :w], "weight0": weights[:, 0:1, :h, :w]}
        return out[..., :h, :w], aux


class FGInference(nn.Module):
    """Export wrapper returning only the interpolated frame."""

    def __init__(self, net):
        super().__init__()
        self.net = net

    def forward(self, f0, f1):
        return self.net(f0, f1)[0]


def count_params(model):
    return sum(p.numel() for p in model.parameters())


def macs_per_lowres_pixel(in_ch, mid):
    return in_ch * mid * 9 + 2 * mid * mid * 9 + mid * 9 * 9


def charbonnier(pred, target, eps=1e-3):
    return torch.sqrt((pred - target) ** 2 + eps * eps).mean()


def _census(img, patch):
    gray = 0.299 * img[:, 0:1] + 0.587 * img[:, 1:2] + 0.114 * img[:, 2:3]
    eye = torch.eye(patch * patch, device=img.device, dtype=img.dtype).reshape(patch * patch, 1, patch, patch)
    diff = F.conv2d(gray, eye, padding=patch // 2) - gray
    return diff / torch.sqrt(0.81 + diff * diff)


def census_loss(pred, target, patch=7):
    """Soft ternary census distance over a patch x patch window (robust to brightness shifts)."""
    d = (_census(pred, patch) - _census(target, patch)) ** 2
    dist = (d / (0.1 + d)).mean(dim=1, keepdim=True)
    b = patch // 2
    return dist[..., b:-b, b:-b].mean()


def psnr(a, b):
    mse = ((a.float() - b.float()) ** 2).mean(dim=(1, 2, 3))
    return 10.0 * torch.log10(1.0 / mse.clamp_min(1e-10))


def interpolation_error(a, b):
    """Middlebury-style interpolation error: RMS difference on the 0-255 scale."""
    return ((a.float() - b.float()) ** 2).mean(dim=(1, 2, 3)).sqrt() * 255.0


def ssim(a, b):
    a, b = a.float(), b.float()
    channels = a.shape[1]
    coords = torch.arange(11, dtype=torch.float32, device=a.device) - 5
    g = torch.exp(-(coords ** 2) / (2 * 1.5 ** 2))
    g = g / g.sum()
    window = (g[:, None] * g[None, :]).expand(channels, 1, 11, 11).contiguous()
    mu_a, mu_b = F.conv2d(a, window, groups=channels), F.conv2d(b, window, groups=channels)
    var_a = F.conv2d(a * a, window, groups=channels) - mu_a ** 2
    var_b = F.conv2d(b * b, window, groups=channels) - mu_b ** 2
    cov = F.conv2d(a * b, window, groups=channels) - mu_a * mu_b
    c1, c2 = 0.01 ** 2, 0.03 ** 2
    score = ((2 * mu_a * mu_b + c1) * (2 * cov + c2)) / ((mu_a ** 2 + mu_b ** 2 + c1) * (var_a + var_b + c2))
    return score.mean(dim=(1, 2, 3))


def quantize(x):
    return (x.clamp(0.0, 1.0) * 255.0).round() / 255.0


def augment_triplets(f0, gt, f1):
    """Same random flip and transpose for all three frames of each sample, plus random time reversal."""
    n = f0.shape[0]
    stack = torch.cat([f0, gt, f1], dim=1)
    flags = torch.rand(n, 4, device=stack.device) < 0.5
    stack = torch.where(flags[:, 0, None, None, None], stack.flip(-1), stack)
    stack = torch.where(flags[:, 1, None, None, None], stack.flip(-2), stack)
    stack = torch.where(flags[:, 2, None, None, None], stack.transpose(-1, -2), stack)
    f0, gt, f1 = stack[:, 0:3], stack[:, 3:6], stack[:, 6:9]
    swap = flags[:, 3, None, None, None]
    return torch.where(swap, f1, f0), gt, torch.where(swap, f0, f1)

## 5. Library: dataset discovery, triplets and patch extraction

Written to `code/fg_data.py`.

* `find_dataset_root` supports both mount layouts (`/kaggle/input/<slug>` and `/kaggle/input/datasets/<owner>/<slug>`); `sampled_walk` lists at most `list_cap_files` images per dataset from randomly chosen directories, always listing a directory completely.
* `is_clean_colour` drops degraded copies (LR, bicubic, blurred) and non-colour passes (flow, occlusion, invalid, depth, albedo).
* `build_triplets` sorts each folder's frames in natural order and takes consecutive windows; `read_test_list` finds the Vimeo-90K official test list.
* `extract_triplet_patch` decodes a triplet in a worker process and returns one crop at the same position in all three frames, retrying flat crops.

In [ ]:
%%writefile /kaggle/working/code/fg_data.py
"""Dataset discovery, triplet building, splitting and triplet-patch extraction for the Kaggle FG notebook."""
import hashlib
import os
import random
import re
from pathlib import Path

import numpy as np
from PIL import Image

IMG_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff", ".ppm"}
# Path tokens that mark degraded copies (LR, blurred, compressed) or non-colour passes.
LOW_QUALITY_TOKEN = re.compile(r"^(lr\d*|lq|low|lowres|bicubic|bix\d|bdx\d|lrbi|lrbd|x\d|blur|blurred|noisy|compressed|jpeg)$")
NON_COLOUR_TOKENS = {"occlusions", "occlusion", "invalid", "flow", "viz", "mask", "masks", "depth", "albedo",
                     "shading", "disparities", "segmentation", "camdata", "edges", "normals"}
HR_TOKENS = {"hr", "gt", "original", "groundtruth"}


def tokens(text):
    return [t for t in re.split(r"[_\-\s.]+", text.lower()) if t]


def path_tokens(path, root):
    rel = Path(path).relative_to(root)
    parts = list(rel.parts[:-1]) + [Path(rel.parts[-1]).stem]
    return [t for part in parts for t in tokens(part)]


def is_clean_colour(path, root):
    return not any(LOW_QUALITY_TOKEN.match(t) or t in NON_COLOUR_TOKENS for t in path_tokens(path, root))


def has_hr_marker(path, root):
    return any(t in HR_TOKENS for t in path_tokens(path, root))


def find_dataset_root(input_root, owner_slug):
    """Return the mount folder of a Kaggle dataset under either supported layout, or None."""
    owner, slug = owner_slug.split("/")
    input_root = Path(input_root)
    for candidate in (input_root / slug, input_root / "datasets" / owner / slug, input_root / "datasets" / slug):
        if candidate.is_dir():
            return candidate
    for depth_one in input_root.iterdir() if input_root.is_dir() else []:
        if depth_one.is_dir():
            for depth_two in [depth_one] + [p for p in depth_one.iterdir() if p.is_dir()]:
                if depth_two.name == slug:
                    return depth_two
    return None


def sampled_walk(root, max_files, seed):
    """List image files, expanding a random directory from the frontier each step.

    Random frontier expansion samples many sequences early instead of exhausting one branch,
    so a capped listing stays diverse while touching only a fraction of a large mount.
    Each directory is listed completely, so a listed sequence folder is never cut in half.
    """
    rng = random.Random(seed)
    frontier, files, dirs_listed = [str(root)], [], 0
    while frontier and len(files) < max_files:
        i = rng.randrange(len(frontier))
        frontier[i], frontier[-1] = frontier[-1], frontier[i]
        current = frontier.pop()
        try:
            entries = list(os.scandir(current))
        except OSError:
            continue
        dirs_listed += 1
        for entry in entries:
            if entry.name.startswith("."):
                continue
            if entry.is_dir(follow_symlinks=False):
                frontier.append(entry.path)
            elif os.path.splitext(entry.name)[1].lower() in IMG_EXTS:
                files.append(entry.path)
    return sorted(files), dirs_listed, len(frontier) == 0


def natural_key(path):
    return [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", Path(path).name.lower())]


def build_triplets(files, max_per_folder, seed):
    """Consecutive (t-1, t-0.5, t) windows inside each folder, at most max_per_folder per folder."""
    rng = random.Random(seed)
    folders = {}
    for f in files:
        folders.setdefault(str(Path(f).parent), []).append(f)
    triplets = []
    for folder, frames in sorted(folders.items()):
        frames = sorted(frames, key=natural_key)
        windows = [tuple(frames[i:i + 3]) for i in range(len(frames) - 2)]
        if len(windows) > max_per_folder:
            windows = rng.sample(windows, max_per_folder)
        triplets.extend((folder, w) for w in windows)
    return triplets


def sequence_name(path):
    """Sequence folder of a video frame, skipping a trailing HR/GT marker folder (calendar/GT/x.png)."""
    parent = Path(path).parent
    return parent.parent.name if parent.name.lower() in HR_TOKENS else parent.name


def group_key(folder, level):
    """Split unit: 1 = the sequence folder itself, 2 = its parent (video)."""
    p = Path(folder)
    return p.name if level == 1 else p.parent.name


def in_holdout(dataset, key, permille, salt=""):
    digest = hashlib.md5(f"{salt}{dataset}/{key}".encode()).hexdigest()
    return int(digest, 16) % 1000 < permille


def read_test_list(root):
    """Vimeo-90K official test list entries such as '00001/0389', if the mirror ships one."""
    for pattern in ("*testlist*.txt", "*/*testlist*.txt", "*/*/*testlist*.txt"):
        found = sorted(Path(root).glob(pattern))
        if found:
            return found[0], {line.strip() for line in found[0].read_text().splitlines() if line.strip()}
    return None, set()


def interleave_by_group(items, groups, seed):
    """Round-robin over shuffled groups so a capped selection spans many videos."""
    rng = random.Random(seed)
    buckets = {}
    for item, g in zip(items, groups):
        buckets.setdefault(g, []).append(item)
    order = list(buckets)
    rng.shuffle(order)
    for g in order:
        rng.shuffle(buckets[g])
    out, depth = [], 0
    while len(out) < len(items):
        for g in order:
            if depth < len(buckets[g]):
                out.append(buckets[g][depth])
        depth += 1
    return out


def load_rgb(path):
    with Image.open(path) as im:
        return np.asarray(im.convert("RGB"))


def image_size(path):
    try:
        with Image.open(path) as im:
            return im.size
    except Exception:
        return None


def motion_proxy(frames):
    """Mean absolute difference between frames t-1 and t on 64x64 grey thumbnails (0-255)."""
    try:
        a, b = (np.asarray(Image.open(f).convert("L").resize((64, 64)), dtype=np.float32) for f in (frames[0], frames[2]))
        return float(np.abs(a - b).mean())
    except Exception:
        return float("nan")


def extract_triplet_patch(task):
    """Worker: decode a triplet and return one random p x p crop at the same place in all frames."""
    frames, p, seed, min_std = task
    try:
        imgs = [load_rgb(f) for f in frames]
    except Exception:
        return frames, None
    h, w = imgs[0].shape[:2]
    if any(im.shape[:2] != (h, w) for im in imgs) or h < p or w < p:
        return frames, None
    rng = np.random.default_rng(seed)
    for _ in range(3):
        y, x = int(rng.integers(0, h - p + 1)), int(rng.integers(0, w - p + 1))
        crop = np.stack([im[y:y + p, x:x + p] for im in imgs])
        if crop[1].mean(axis=2).std() >= min_std:
            return frames, crop
    return frames, crop

## 6. DDP training script

Written to `code/train_fg.py` and launched in section 13 with `torch.distributed.run` (one process per GPU, NCCL).

* Data: whole batches from the memory-mapped triplet cache through a `DataLoader` with a distributed batch sampler, pinned memory, persistent workers and prefetching; crop and augmentation on the GPU.
* Step: FP16 autocast for the CNN, FP32 warping and loss, gradient scaling, clipping at 1.0, AdamW, EMA weights on rank 0.
* Every `sync_every` steps one all-reduce shares elapsed time, the validation flag and per-GPU peak memory, so both ranks use the same learning rate and stop on the same step.
* Every `val_interval_min` minutes rank 0 validates the EMA weights against the frame-average and frame-copy baselines, saves `fg_last.pt` and, on improvement, `fg_best_ema.pt`, and prints a `ROUND` line.

In [ ]:
%%writefile /kaggle/working/code/train_fg.py
"""DDP training entry point for NeuralFG, launched by the notebook through torch.distributed.run.

Rank 0 prints machine-readable lines that the notebook turns into a live progress bar and plots:
PROGRESS {json} every few seconds, ROUND {json} after each validation round, DONE {json} at the end.
"""
import argparse
import copy
import json
import math
import os
import subprocess
import sys
import time
from datetime import timedelta

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import BatchSampler, DataLoader, Dataset, DistributedSampler

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from fg_lib import NeuralFG, augment_triplets, census_loss, charbonnier, interpolation_error, psnr, quantize, ssim  # noqa: E402


class TripletCache(Dataset):
    """Returns whole batches of uint8 triplets (B, 3, P, P, 3) from a memory-mapped .npy cache."""

    def __init__(self, path, count):
        self.path, self.count, self.data = path, count, None

    def __len__(self):
        return self.count

    def __getitem__(self, indices):
        if self.data is None:
            self.data = np.load(self.path, mmap_mode="r")
        return torch.from_numpy(np.ascontiguousarray(self.data[np.sort(np.asarray(indices))]))


def lr_at(progress, cfg):
    """Linear warm-up then cosine decay, driven by the fraction of the wall-clock budget used."""
    if progress < cfg["warmup_frac"]:
        return cfg["lr"] * max(progress / cfg["warmup_frac"], 0.01)
    t = min(1.0, (progress - cfg["warmup_frac"]) / (1.0 - cfg["warmup_frac"]))
    return cfg["min_lr"] + 0.5 * (cfg["lr"] - cfg["min_lr"]) * (1.0 + math.cos(math.pi * t))


def to_frames(batch, device, crop=None):
    """uint8 (B, 3, P, P, 3) -> three float NCHW frames, optionally randomly cropped (one offset per batch)."""
    x = batch.to(device, non_blocking=True).permute(0, 1, 4, 2, 3).float().div_(255.0)
    size = x.shape[-1]
    if crop is not None and crop < size:
        oy, ox = np.random.randint(0, size - crop + 1, size=2)
        x = x[..., oy:oy + crop, ox:ox + crop]
    return x[:, 0], x[:, 1], x[:, 2]


def gpu_utilisation():
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10).stdout
        return [float(v) for v in out.split()]
    except Exception:
        return []


@torch.no_grad()
def validate(model, val_triplets, device, batch):
    model.eval()
    rows = {"psnr": [], "ssim": [], "ie": [], "avg_psnr": [], "copy_psnr": []}
    for i in range(0, val_triplets.shape[0], batch):
        f0, gt, f1 = to_frames(torch.from_numpy(val_triplets[i:i + batch]), device)
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(f0, f1)[0]
        pred = quantize(pred.float())
        rows["psnr"].append(psnr(pred, gt))
        rows["ssim"].append(ssim(pred, gt))
        rows["ie"].append(interpolation_error(pred, gt))
        rows["avg_psnr"].append(psnr(quantize((f0 + f1) / 2), gt))
        rows["copy_psnr"].append(psnr(f0, gt))
    return {k: torch.cat(v).mean().item() for k, v in rows.items()}


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    cfg = json.load(open(parser.parse_args().config))

    dist.init_process_group("nccl", timeout=timedelta(minutes=30))
    rank, world = dist.get_rank(), dist.get_world_size()
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    device = torch.device("cuda", local_rank)
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"] + rank)
    torch.backends.cudnn.benchmark = True

    model = NeuralFG(in_ch=6, mid=cfg["mid_channels"], downsample=cfg["downsample"]).to(device)
    ema = copy.deepcopy(model).eval().requires_grad_(False) if rank == 0 else None
    ddp = DDP(model, device_ids=[local_rank], broadcast_buffers=False, gradient_as_bucket_view=True)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], betas=(0.9, 0.999), weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda")
    val_triplets = np.load(cfg["val_cache"])[: cfg["val_count"]] if rank == 0 else None

    sampler = DistributedSampler(range(cfg["train_count"]), num_replicas=world, rank=rank,
                                 shuffle=True, seed=cfg["seed"], drop_last=True)
    loader = DataLoader(TripletCache(cfg["train_cache"], cfg["train_count"]),
                        sampler=BatchSampler(sampler, cfg["batch_per_gpu"], drop_last=True), batch_size=None,
                        num_workers=cfg["workers_per_rank"], pin_memory=True,
                        persistent_workers=True, prefetch_factor=4)

    start = time.time()
    last_val = last_progress = round_start = start
    sync = torch.zeros(2 + world, device=device)
    train_acc = torch.zeros(2, device=device)
    step = epoch = round_steps = round_samples = 0
    best, history, done, lr = -1.0, [], False, lr_at(0.0, cfg)
    while not done:
        sampler.set_epoch(epoch)
        for batch in loader:
            if step % cfg["sync_every"] == 0:
                # One collective agrees on time, validation and per-GPU memory, so every rank
                # uses the same learning rate and stops at the same step.
                sync.zero_()
                if rank == 0:
                    elapsed = time.time() - start
                    sync[0] = elapsed
                    sync[1] = float(elapsed - (last_val - start) >= cfg["val_interval_s"] or elapsed >= cfg["train_seconds"])
                sync[2 + rank] = torch.cuda.max_memory_allocated(device) / 2 ** 30
                dist.all_reduce(sync)
                elapsed, progress = sync[0].item(), min(1.0, sync[0].item() / cfg["train_seconds"])
                mem_gb = [round(v, 2) for v in sync[2:].tolist()]
                lr = lr_at(progress, cfg)
                for group in opt.param_groups:
                    group["lr"] = lr
                if sync[1].item() > 0.5 and round_steps > 0:
                    dist.all_reduce(train_acc)
                    if rank == 0:
                        now = time.time()
                        row = {"round": len(history), "elapsed_min": round(elapsed / 60, 2), "step": step, "epoch": epoch,
                               "train_loss": train_acc[0].item() / (round_steps * world),
                               "train_psnr": train_acc[1].item() / (round_steps * world),
                               "lr": lr, "triplets_per_s": round_samples * world / (now - round_start),
                               "mem_gb": mem_gb, "util_pct": gpu_utilisation()}
                        row.update({"val_" + k: v for k, v in validate(ema, val_triplets, device, cfg["val_batch"]).items()})
                        history.append(row)
                        state = {"model": model.state_dict(), "ema": ema.state_dict(), "optimizer": opt.state_dict(),
                                 "scaler": scaler.state_dict(), "step": step, "elapsed_s": elapsed, "cfg": cfg}
                        torch.save(state, os.path.join(cfg["weights_dir"], "fg_last.pt"))
                        if row["val_psnr"] > best:
                            best = row["val_psnr"]
                            torch.save({"ema": ema.state_dict(), "round": row["round"], "val_psnr": best, "cfg": cfg},
                                       os.path.join(cfg["weights_dir"], "fg_best_ema.pt"))
                        row["best_val_psnr"] = best
                        print("ROUND " + json.dumps(row), flush=True)
                        last_val = time.time()
                    dist.barrier()
                    train_acc.zero_()
                    round_steps = round_samples = 0
                    round_start = time.time()
                if progress >= 1.0:
                    done = True
                    break
                if rank == 0 and time.time() - last_progress >= cfg["progress_interval_s"]:
                    print("PROGRESS " + json.dumps({
                        "progress": progress, "step": step, "elapsed_min": round(elapsed / 60, 2), "lr": lr,
                        "loss": train_acc[0].item() / max(round_steps, 1),
                        "triplets_per_s": round_samples * world / max(time.time() - round_start, 1e-6), "mem_gb": mem_gb}), flush=True)
                    last_progress = time.time()

            f0, gt, f1 = augment_triplets(*to_frames(batch, device, cfg["crop"]))
            with torch.autocast("cuda", dtype=torch.float16):
                pred = ddp(f0, f1)[0]
            pred = pred.float()
            loss = charbonnier(pred, gt) + cfg["w_census"] * census_loss(pred, gt)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
            scaler.step(opt)
            scaler.update()
            if rank == 0:
                decay = min(cfg["ema_decay"], (1 + step) / (10 + step))
                with torch.no_grad():
                    for e, p in zip(ema.parameters(), model.parameters()):
                        e.lerp_(p, 1.0 - decay)
            with torch.no_grad():
                train_acc[0] += loss.detach()
                train_acc[1] += psnr(pred.detach().clamp(0, 1), gt).mean()
            step += 1
            round_steps += 1
            round_samples += f0.shape[0]
        epoch += 1

    if rank == 0:
        with open(os.path.join(cfg["metrics_dir"], "train_history.json"), "w") as f:
            json.dump(history, f, indent=2)
        print("DONE " + json.dumps({"steps": step, "epochs": epoch, "train_seconds": round(time.time() - start, 1),
                                    "best_val_psnr": best}), flush=True)
    dist.barrier()
    dist.destroy_process_group()


if __name__ == "__main__":
    main()

## 7. Library import and model checks

Imports the generated modules and runs three CPU checks: the model output shape on an odd size (padding path), that an untrained model returns exactly the frame average (zero-initialised head and an exact identity warp), and that a constant flow of +3 pixels reproduces a shifted image. The table lists parameters and multiply-accumulates for the RGB build trained here and the 10-channel engine build in `RESEARCH.md`. Expected output: all checks near 1e-6 and about 22.9K parameters.

In [ ]:
sys.path.insert(0, str(DIRS["code"]))
from fg_data import (
    build_triplets,
    extract_triplet_patch,
    find_dataset_root,
    group_key,
    has_hr_marker,
    image_size,  # noqa: E402
    in_holdout,
    interleave_by_group,
    is_clean_colour,
    load_rgb,
    motion_proxy,
    read_test_list,
    sampled_walk,
    sequence_name,
)
from fg_lib import (
    FGInference,
    NeuralFG,
    backward_warp,
    census_loss,
    charbonnier,
    count_params,  # noqa: E402
    interpolation_error,
    macs_per_lowres_pixel,
    psnr,
    quantize,
    ssim,
)

torch.manual_seed(0)
check_model = NeuralFG(mid=CFG["mid_channels"], downsample=CFG["downsample"]).eval()
a, b = torch.rand(2, 3, 50, 70), torch.rand(2, 3, 50, 70)
with torch.no_grad():
    out, aux = check_model(a, b)
    average_error = (out - (a + b) / 2).abs().max().item()
    shifted = backward_warp(a, torch.full((2, 2, 50, 70), 3.0))
    shift_error = (shifted[..., :-3, :-3] - a[..., 3:, 3:]).abs().max().item()
rows = []
for name, in_ch in (
    ("RGB build (this notebook)", 6),
    ("engine build (RGB + MV + 2 depths)", 10),
):
    macs = macs_per_lowres_pixel(in_ch, CFG["mid_channels"])
    rows.append(
        {
            "variant": name,
            "params": count_params(NeuralFG(in_ch=in_ch, mid=CFG["mid_channels"])),
            "gmac_1080p": round(macs * (1920 // 4) * (1080 // 4) / 1e9, 3),
        }
    )
print(pd.DataFrame(rows).to_string(index=False))
print(
    pd.Series(
        {
            "output_shape": tuple(out.shape),
            "untrained_vs_average_max_abs": f"{average_error:.2e}",
            "warp_shift_max_abs": f"{shift_error:.2e}",
        }
    ).to_string()
)
assert out.shape == a.shape and average_error < 1e-5 and shift_error < 1e-5
del check_model

## 8. Dataset discovery, triplets and split

Lists every registered dataset in parallel threads, filters files, builds triplets and assigns splits:

* Vimeo triplet: official test list entries go to `test`; without a list, about 3 percent of videos are hashed into `test`. Vimeo septuplet: official test septuplets are marked `excluded`.
* All training datasets: about 3 percent of groups go to `val`, the rest to `train`. Vid4 triplets go to `test` as a zero-shot set.

Expected output: a per-dataset table (found, files listed, triplets by split, test list used), a stacked bar chart, and `metrics/catalogue.csv`.

In [ ]:
def list_source(src):
    start = time.time()
    root = find_dataset_root(INPUT_ROOT, src["slug"])
    if root is None:
        return src, None, [], 0, False, time.time() - start
    files, dirs_listed, complete = sampled_walk(
        root, CFG["list_cap_files"], CFG["seed"]
    )
    return src, root, files, dirs_listed, complete, time.time() - start


with stage("discovery"):
    listed = []
    with ThreadPoolExecutor(max_workers=len(SOURCES)) as pool:
        futures = [pool.submit(list_source, src) for src in SOURCES]
        for future in tqdm(
            as_completed(futures), total=len(futures), desc="Listing datasets"
        ):
            listed.append(future.result())
    listed.sort(key=lambda r: SOURCES.index(r[0]))

    records, summary = [], []
    for src, root, files, dirs_listed, complete, seconds in listed:
        row = {
            "dataset": src["name"],
            "role": src["role"],
            "found": root is not None,
            "listed": len(files),
            "dirs_listed": dirs_listed,
            "listing_complete": complete,
            "seconds": round(seconds, 1),
            "test_list": None,
        }
        if root is None:
            summary.append(row)
            continue
        kept = [f for f in files if is_clean_colour(f, root)]
        if src["role"] == "video_test" and any(has_hr_marker(f, root) for f in kept):
            kept = [f for f in kept if has_hr_marker(f, root)]
        test_list_path, test_list = (
            read_test_list(root) if src["name"].startswith("vimeo") else (None, set())
        )
        row["test_list"] = (
            str(test_list_path.relative_to(root)) if test_list_path else None
        )
        for folder, frames in build_triplets(kept, src["max_per_folder"], CFG["seed"]):
            key = group_key(folder, src["group_level"])
            clip = "/".join(Path(folder).parts[-2:])
            if src["role"] == "video_test":
                split = "test"
            elif src["name"] == "vimeo_triplet" and (
                clip in test_list
                if test_list
                else in_holdout(src["name"], key, CFG["test_permille"], "test")
            ):
                split = "test"
            elif src["name"] == "vimeo_septuplet" and clip in test_list:
                split = "excluded"
            else:
                split = (
                    "val"
                    if in_holdout(src["name"], key, CFG["val_permille"])
                    else "train"
                )
            records.append(
                {
                    "dataset": src["name"],
                    "role": src["role"],
                    "group": key,
                    "split": split,
                    "f0": frames[0],
                    "gt": frames[1],
                    "f1": frames[2],
                }
            )
        summary.append(row)
    catalogue = pd.DataFrame(
        records, columns=["dataset", "role", "group", "split", "f0", "gt", "f1"]
    )
    split_counts = catalogue.groupby(["dataset", "split"]).size().unstack(fill_value=0)
    discovery_df = (
        pd.DataFrame(summary).set_index("dataset").join(split_counts).fillna(0)
    )
    discovery_df["groups"] = catalogue.groupby("dataset")["group"].nunique()

print(discovery_df.to_string())
catalogue.to_csv(DIRS["metrics"] / "catalogue.csv", index=False)
discovery_df.to_csv(DIRS["metrics"] / "discovery.csv")
missing = [s["name"] for s, root, *_ in listed if root is None]
if missing:
    log.warning("datasets not attached (skipped): %s", ", ".join(missing))
fig, ax = plt.subplots(figsize=(9, 4))
split_counts.reindex(
    columns=[c for c in ["train", "val", "test", "excluded"] if c in split_counts]
).plot.barh(stacked=True, ax=ax)
ax.set_xlabel("triplets")
ax.set_title("Triplets by dataset and split")
show(fig, "dataset_counts")
TRAIN_SOURCES = [
    s
    for s in SOURCES
    if s["role"] == "train"
    and ((catalogue["dataset"] == s["name"]) & (catalogue["split"] == "train")).any()
]
assert (
    TRAIN_SOURCES
), "No training triplets found. Attach at least one training dataset listed in the overview."

## 9. Exploratory analysis

For a stratified sample of training triplets (threads, with a progress bar): frame sizes from image headers and a motion proxy (mean absolute difference between t-1 and t on 64x64 grey thumbnails, 0-255 scale), per dataset. Larger values mean larger or faster motion, which is harder to interpolate. Then a preview of four triplets with their difference images. Expected output: a summary table, distribution plots, the preview, and `metrics/eda_triplets.csv`.

In [ ]:
with stage("eda"):
    train_cat = catalogue[catalogue["split"] == "train"]
    per_source = max(1, CFG["eda_samples"] // max(1, train_cat["dataset"].nunique()))
    eda = train_cat.groupby("dataset", group_keys=False).apply(
        lambda d: d.sample(min(len(d), per_source), random_state=CFG["seed"])
    )
    frame_lists = [(r.f0, r.gt, r.f1) for r in eda.itertuples()]
    with ThreadPoolExecutor(max_workers=16) as pool:
        sizes = list(
            tqdm(
                pool.map(image_size, eda["gt"]),
                total=len(eda),
                desc="Reading frame headers",
            )
        )
        motion = list(
            tqdm(
                pool.map(motion_proxy, frame_lists), total=len(eda), desc="Motion proxy"
            )
        )
    eda = eda.assign(
        width=[s[0] if s else np.nan for s in sizes],
        height=[s[1] if s else np.nan for s in sizes],
        motion=motion,
    )

eda_summary = eda.groupby("dataset").agg(
    n=("gt", "size"),
    median_w=("width", "median"),
    median_h=("height", "median"),
    motion_median=("motion", "median"),
    motion_p90=("motion", lambda m: m.quantile(0.9)),
)
print(eda_summary.round(2).to_string())
eda.drop(columns=["f0", "gt", "f1"]).to_csv(
    DIRS["metrics"] / "eda_triplets.csv", index=False
)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2))
for name, group in eda.groupby("dataset"):
    axes[0].hist(group["motion"].dropna(), bins=40, alpha=0.5, label=name)
axes[0].set_xlabel("motion proxy (mean |t-1 - t|, 0-255)")
axes[0].set_ylabel("triplets")
axes[0].legend()
axes[0].set_title("Motion distribution by dataset")
eda.boxplot(column="width", by="dataset", ax=axes[1], rot=30)
axes[1].set_title("Frame width by dataset")
axes[1].set_xlabel("")
fig.suptitle("")
show(fig, "eda_motion_sizes")

preview = eda.sort_values("motion").iloc[np.linspace(0, len(eda) - 1, 4).astype(int)]
fig, axes = plt.subplots(4, 4, figsize=(15, 9))
for row_axes, r in zip(axes, preview.itertuples()):
    frames = [load_rgb(p) for p in (r.f0, r.gt, r.f1)]
    diff = np.abs(frames[0].astype(np.float32) - frames[2].astype(np.float32)).mean(
        axis=2
    )
    for ax, img, title in zip(
        row_axes,
        frames + [diff],
        ["t-1", "GT t-0.5", "t", f"|t-1 - t| (motion {r.motion:.1f})"],
    ):
        ax.imshow(img, cmap="magma" if img.ndim == 2 else None)
        ax.set_title(f"{r.dataset}: {title}", fontsize=8)
        ax.axis("off")
fig.suptitle("Triplet previews, from least to most motion")
show(fig, "eda_triplet_preview")

## 10. Triplet cache build

Decodes training triplets once, in a process pool over all CPU cores, into `cache/train_triplets.npy` (uint8, N x 3 x 256 x 256 x 3, memory-mapped) and `cache/val_triplets.npy`.

* Size: the smallest of `cache_gb`, 35 percent of available RAM (so the page cache can hold it for both GPU processes) and 45 percent of free disk.
* Balance: triplets are allocated by quota; when a dataset has fewer triplets than its share, the rest is redistributed to the others. Triplets are picked round-robin across videos and the task list is shuffled, so an early stop keeps the mix.
* Time cap: the build stops after `cache_build_min` minutes and keeps what is filled.

Expected output: live progress bars, a table of cached triplets per dataset against the allocation, and a bar chart.

In [ ]:
def allocate(total, quotas, available):
    """Split total by quota, capped by availability, redistributing any shortfall."""
    alloc = {k: 0 for k in quotas}
    remaining, open_keys = total, set(quotas)
    while remaining > 0 and open_keys:
        weight = sum(quotas[k] for k in open_keys)
        step = {k: int(remaining * quotas[k] / weight) for k in open_keys}
        if not any(step.values()):
            break
        for k in list(open_keys):
            take = min(step[k], available[k] - alloc[k])
            alloc[k] += take
            remaining -= take
            if alloc[k] >= available[k]:
                open_keys.discard(k)
    return alloc


P = CFG["cache_patch"]
triplet_bytes = 3 * P * P * 3
budget_bytes = min(
    CFG["cache_gb"] * 1e9,
    0.35 * psutil.virtual_memory().available,
    0.45 * shutil.disk_usage(WORK).free,
)
n_target = int(budget_bytes // triplet_bytes)
train_rows = catalogue[catalogue["split"] == "train"]
pools = {
    s["name"]: train_rows[train_rows["dataset"] == s["name"]] for s in TRAIN_SOURCES
}
alloc = allocate(
    n_target,
    {s["name"]: s["quota"] for s in TRAIN_SOURCES},
    {k: len(v) for k, v in pools.items()},
)

tasks, task_source = [], {}
for name, rows in pools.items():
    frames = list(zip(rows["f0"], rows["gt"], rows["f1"]))
    for trip in interleave_by_group(frames, rows["group"].tolist(), CFG["seed"])[
        : alloc[name]
    ]:
        tasks.append((trip, P, CFG["seed"] + len(tasks), CFG["min_patch_std"]))
        task_source[trip] = name
random.Random(CFG["seed"]).shuffle(tasks)
val_rows = catalogue[catalogue["split"] == "val"]
val_frames = interleave_by_group(
    list(zip(val_rows["f0"], val_rows["gt"], val_rows["f1"])),
    val_rows["group"].tolist(),
    CFG["seed"],
)
val_tasks = [
    (trip, P, CFG["seed"] + i, CFG["min_patch_std"])
    for i, trip in enumerate(val_frames[: CFG["val_triplets"]])
]
capacity = min(n_target, len(tasks))
log.info(
    "cache target %d triplets (%.2f GB), %d decode tasks",
    capacity,
    capacity * triplet_bytes / 1e9,
    len(tasks),
)


def fill_cache(task_list, capacity, out, desc, deadline):
    filled, failed, counts = 0, 0, {}
    bar = tqdm(total=capacity, desc=desc, unit="triplet")
    with Pool(os.cpu_count()) as pool:
        for frames, crop in pool.imap_unordered(
            extract_triplet_patch, task_list, chunksize=4
        ):
            if crop is None:
                failed += 1
                continue
            out[filled] = crop
            filled += 1
            source = task_source.get(frames, "val")
            counts[source] = counts.get(source, 0) + 1
            bar.update(1)
            if filled >= capacity or time.time() > deadline:
                break
    bar.close()
    return filled, failed, counts


with stage("cache_build"):
    deadline = time.time() + CFG["cache_build_min"] * 60
    train_cache = np.lib.format.open_memmap(
        DIRS["cache"] / "train_triplets.npy",
        mode="w+",
        dtype=np.uint8,
        shape=(capacity, 3, P, P, 3),
    )
    train_count, train_failed, train_counts = fill_cache(
        tasks, capacity, train_cache, "Train triplets", deadline
    )
    train_cache.flush()
    del train_cache
    val_cache = np.zeros((len(val_tasks), 3, P, P, 3), dtype=np.uint8)
    val_count, val_failed, _ = fill_cache(
        val_tasks, len(val_tasks), val_cache, "Validation triplets", time.time() + 600
    )
    np.save(DIRS["cache"] / "val_triplets.npy", val_cache[:val_count])
    del val_cache

cache_df = (
    pd.DataFrame({"allocated": pd.Series(alloc), "cached": pd.Series(train_counts)})
    .fillna(0)
    .astype(int)
)
cache_df["share_pct"] = (100 * cache_df["cached"] / max(1, train_count)).round(1)
print(cache_df.to_string())
CACHE_INFO = {
    "train_count": train_count,
    "train_capacity": capacity,
    "val_count": val_count,
    "patch": P,
    "train_gb": round(train_count * triplet_bytes / 1e9, 2),
    "decode_failures": train_failed + val_failed,
}
print(pd.Series(CACHE_INFO).to_string())
save_json(
    {**CACHE_INFO, "per_dataset": cache_df.reset_index().to_dict("records")},
    "cache_summary.json",
)
fig, ax = plt.subplots(figsize=(8, 3.8))
cache_df[["allocated", "cached"]].plot.bar(ax=ax, rot=0)
ax.set_ylabel("triplets")
ax.set_title("Triplet cache: allocation versus cached, by dataset")
show(fig, "cache_composition")
assert (
    train_count >= 500 and val_count >= 32
), "Triplet cache is too small; attach more training data or raise cache_build_min."

## 11. Triplet cache inspection

Samples cached triplets to check alignment and content: six triplets with their t-1 to t difference, and the distribution of motion in the cache (mean absolute difference between t-1 and t on the 0-255 scale). Expected output: aligned crops and a motion histogram similar to the EDA one.

In [ ]:
cache_view = np.load(DIRS["cache"] / "train_triplets.npy", mmap_mode="r")
rng = np.random.default_rng(CFG["seed"])
sample_idx = np.sort(
    rng.choice(train_count, size=min(1500, train_count), replace=False)
)
sample = np.asarray(cache_view[sample_idx])
cache_motion = np.abs(
    sample[:, 0].astype(np.float32) - sample[:, 2].astype(np.float32)
).mean(axis=(1, 2, 3))
fig, axes = plt.subplots(6, 4, figsize=(11, 16))
for row_axes, idx in zip(axes, rng.choice(len(sample), size=6, replace=False)):
    diff = np.abs(
        sample[idx, 0].astype(np.float32) - sample[idx, 2].astype(np.float32)
    ).mean(axis=2)
    for ax, img, title in zip(
        row_axes,
        [sample[idx, 0], sample[idx, 1], sample[idx, 2], diff],
        ["t-1", "GT", "t", "|t-1 - t|"],
    ):
        ax.imshow(img, cmap="magma" if img.ndim == 2 else None)
        ax.set_title(title, fontsize=8)
        ax.axis("off")
fig.suptitle("Random cached triplets (256 x 256)")
show(fig, "cache_triplets")
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(cache_motion, bins=50, color="#55A868")
ax.set_xlabel("mean |t-1 - t| (0-255)")
ax.set_ylabel("triplets")
ax.set_title("Motion in the training cache")
show(fig, "cache_motion")
pd.Series(cache_motion).describe().to_frame("cache_motion").to_csv(
    DIRS["metrics"] / "cache_motion.csv"
)
del cache_view, sample

## 12. Batch-size probe and training configuration

Runs full training steps (augmentation, FP16 forward, FP32 warp and losses, backward, AdamW) on GPU 0 for each candidate batch, recording peak VRAM and throughput until memory runs out. The chosen batch is the largest under 90 percent of VRAM, capped at `max_batch_per_gpu`; gradient accumulation stays at 1 because one step already reaches the target effective batch. The cell then scales the learning rate, computes the training time left inside the 8 hour budget and writes `code/train_config.json`. Expected output: the probe table, plots and final settings.

In [ ]:
with stage("batch_probe"):
    device0 = torch.device("cuda", 0)
    total_vram = torch.cuda.get_device_properties(0).total_memory
    probe_model = NeuralFG(mid=CFG["mid_channels"], downsample=CFG["downsample"]).to(
        device0
    )
    probe_opt = torch.optim.AdamW(probe_model.parameters(), lr=1e-4)
    probe_scaler = torch.amp.GradScaler("cuda")
    probe_rows = []
    for bs in tqdm(CFG["batch_candidates"], desc="Batch probe"):
        frames = pred = loss = None
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device0)
        try:
            frames = [
                torch.rand(bs, 3, CFG["crop"], CFG["crop"], device=device0)
                for _ in range(3)
            ]
            timings = []
            for _ in range(4):
                torch.cuda.synchronize(device0)
                t0 = time.time()
                with torch.autocast("cuda", dtype=torch.float16):
                    pred = probe_model(frames[0], frames[2])[0]
                loss = charbonnier(pred.float(), frames[1]) + CFG[
                    "w_census"
                ] * census_loss(pred.float(), frames[1])
                probe_opt.zero_grad(set_to_none=True)
                probe_scaler.scale(loss).backward()
                probe_scaler.step(probe_opt)
                probe_scaler.update()
                torch.cuda.synchronize(device0)
                timings.append(time.time() - t0)
            peak = torch.cuda.max_memory_allocated(device0) / total_vram
            probe_rows.append(
                {
                    "batch": bs,
                    "fits": True,
                    "peak_vram_pct": round(100 * peak, 1),
                    "triplets_per_s_one_gpu": round(bs / np.mean(timings[1:]), 1),
                }
            )
        except torch.cuda.OutOfMemoryError:
            probe_rows.append(
                {
                    "batch": bs,
                    "fits": False,
                    "peak_vram_pct": np.nan,
                    "triplets_per_s_one_gpu": np.nan,
                }
            )
            break
    del probe_model, probe_opt, probe_scaler, frames, pred, loss
    torch.cuda.empty_cache()

probe_df = pd.DataFrame(probe_rows)
print(probe_df.to_string(index=False))
ok = probe_df[
    probe_df["fits"] & (probe_df["peak_vram_pct"] <= 100 * (1 - CFG["vram_headroom"]))
]
assert len(ok), "Even the smallest batch does not fit in GPU memory."
BATCH_PER_GPU = int(min(ok["batch"].max(), CFG["max_batch_per_gpu"]))
EFFECTIVE_BATCH = BATCH_PER_GPU * N_GPUS
LR = min(CFG["max_lr"], CFG["base_lr"] * math.sqrt(EFFECTIVE_BATCH / CFG["base_batch"]))
elapsed_h = (time.time() - NOTEBOOK_START) / 3600
TRAIN_SECONDS = min(
    CFG["train_hours"] * 3600, (RUN_BUDGET_H - CFG["eval_reserve_h"] - elapsed_h) * 3600
)
assert (
    TRAIN_SECONDS > 300
), "Less than 5 minutes left for training inside the run budget."

TRAIN_CFG = {
    "seed": CFG["seed"],
    "mid_channels": CFG["mid_channels"],
    "downsample": CFG["downsample"],
    "crop": CFG["crop"],
    "train_cache": str(DIRS["cache"] / "train_triplets.npy"),
    "train_count": train_count,
    "val_cache": str(DIRS["cache"] / "val_triplets.npy"),
    "val_count": val_count,
    "val_batch": CFG["val_batch"],
    "batch_per_gpu": BATCH_PER_GPU,
    "lr": LR,
    "min_lr": CFG["min_lr"],
    "warmup_frac": CFG["warmup_frac"],
    "weight_decay": CFG["weight_decay"],
    "w_census": CFG["w_census"],
    "ema_decay": CFG["ema_decay"],
    "grad_clip": CFG["grad_clip"],
    "train_seconds": TRAIN_SECONDS,
    "val_interval_s": CFG["val_interval_min"] * 60,
    "progress_interval_s": CFG["progress_interval_s"],
    "sync_every": CFG["sync_every"],
    "workers_per_rank": max(1, (os.cpu_count() or 2) // N_GPUS),
    "weights_dir": str(DIRS["weights"]),
    "metrics_dir": str(DIRS["metrics"]),
}
TRAIN_CFG_PATH = DIRS["code"] / "train_config.json"
TRAIN_CFG_PATH.write_text(json.dumps(TRAIN_CFG, indent=2))
probe_df.to_csv(DIRS["metrics"] / "batch_probe.csv", index=False)
print(
    pd.Series(
        {
            "batch_per_gpu": BATCH_PER_GPU,
            "effective_batch": EFFECTIVE_BATCH,
            "learning_rate": f"{LR:.2e}",
            "train_minutes": round(TRAIN_SECONDS / 60, 1),
            "workers_per_rank": TRAIN_CFG["workers_per_rank"],
        }
    ).to_string()
)

fit_rows = probe_df[probe_df["fits"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(fit_rows["batch"], fit_rows["peak_vram_pct"], "o-")
axes[0].axhline(
    100 * (1 - CFG["vram_headroom"]), ls="--", color="#C44E52", label="headroom limit"
)
axes[0].set_xlabel("batch per GPU")
axes[0].set_ylabel("peak VRAM (%)")
axes[0].legend()
axes[1].plot(
    fit_rows["batch"], fit_rows["triplets_per_s_one_gpu"], "o-", color="#55A868"
)
axes[1].axvline(BATCH_PER_GPU, ls="--", color="grey", label="chosen")
axes[1].set_xlabel("batch per GPU")
axes[1].set_ylabel("triplets per second (one GPU)")
axes[1].legend()
fig.suptitle("Batch-size probe on GPU 0")
show(fig, "batch_probe")

## 13. Distributed training on both GPUs

Launches `code/train_fg.py` with `torch.distributed.run --nproc_per_node=2` and streams its output: `PROGRESS` lines drive the progress bar (percent of the time budget, with loss, learning rate, throughput and per-GPU memory), `ROUND` lines print a one-line summary and redraw the training curves in place, and everything is written to `logs/train_fg.log`. `NCCL_P2P_DISABLE=1` routes the tiny gradient all-reduce (about 90 KB) through host memory, which costs nothing measurable and avoids PCIe peer-to-peer problems on some cloud VMs. Expected output: a progress bar reaching 100 percent, a summary line every `val_interval_min` minutes, and a final `DONE` line.

In [ ]:
def draw_history(axes, rows):
    hist = pd.DataFrame(rows)
    t = hist["elapsed_min"]
    n_gpu = len(hist["mem_gb"].iloc[0])
    util = [u if len(u) == n_gpu else [np.nan] * n_gpu for u in hist["util_pct"]]
    hist = hist.assign(
        **{f"mem_gpu{g}": [m[g] for m in hist["mem_gb"]] for g in range(n_gpu)},
        **{f"util_gpu{g}": [u[g] for u in util] for g in range(n_gpu)},
    )
    panels = [
        ("train loss", [("train_loss", "train")]),
        (
            "validation PSNR (dB)",
            [
                ("val_psnr", "NeuralFG (EMA)"),
                ("val_avg_psnr", "frame average"),
                ("val_copy_psnr", "copy t-1"),
            ],
        ),
        ("learning rate", [("lr", "lr")]),
        ("throughput (triplets/s, both GPUs)", [("triplets_per_s", "throughput")]),
        ("peak GPU memory (GB)", [(f"mem_gpu{g}", f"GPU {g}") for g in range(n_gpu)]),
        (
            "GPU utilisation at round end (%)",
            [(f"util_gpu{g}", f"GPU {g}") for g in range(n_gpu)],
        ),
    ]
    for ax, (title, series) in zip(axes.flat, panels):
        ax.clear()
        for column, label in series:
            ax.plot(t, hist[column], "o-", ms=3, label=label)
        ax.set_title(title, fontsize=10)
        ax.set_xlabel("minutes")
        if len(series) > 1:
            ax.legend(fontsize=8)
    axes.flat[0].figure.tight_layout()


env = dict(os.environ, PYTHONUNBUFFERED="1", NCCL_P2P_DISABLE="1", OMP_NUM_THREADS="2")
cmd = [
    sys.executable,
    "-m",
    "torch.distributed.run",
    "--standalone",
    f"--nproc_per_node={N_GPUS}",
    str(DIRS["code"] / "train_fg.py"),
    "--config",
    str(TRAIN_CFG_PATH),
]
history, done_info = [], {}
live_fig, live_axes = plt.subplots(2, 3, figsize=(16, 7.5))
live_handle = display(live_fig, display_id=True)
bar = tqdm(
    total=100.0,
    desc="Training (% of time budget)",
    bar_format="{l_bar}{bar}| {n:.1f}/100 [{elapsed}<{remaining}{postfix}]",
)
with stage("training"), open(DIRS["logs"] / "train_fg.log", "w") as train_log:
    proc = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=env,
    )
    for line in proc.stdout:
        train_log.write(line)
        train_log.flush()
        kind, _, payload = line.partition(" ")
        if kind == "PROGRESS":
            msg = json.loads(payload)
            bar.n = round(100 * msg["progress"], 1)
            bar.set_postfix(
                loss=f"{msg['loss']:.4f}",
                lr=f"{msg['lr']:.1e}",
                trip_s=f"{msg['triplets_per_s']:.0f}",
                mem_gb="/".join(f"{m:.1f}" for m in msg["mem_gb"]),
            )
        elif kind == "ROUND":
            row = json.loads(payload)
            history.append(row)
            print(
                f"round {row['round']:3d} | {row['elapsed_min']:7.1f} min | loss {row['train_loss']:.4f} | "
                f"val PSNR {row['val_psnr']:.3f} dB (average {row['val_avg_psnr']:.3f}, copy {row['val_copy_psnr']:.3f}) | "
                f"SSIM {row['val_ssim']:.4f} | IE {row['val_ie']:.2f} | {row['triplets_per_s']:.0f} triplets/s | "
                f"mem GB {row['mem_gb']} | util % {row['util_pct']}",
                flush=True,
            )
            draw_history(live_axes, history)
            live_handle.update(live_fig)
        elif kind == "DONE":
            done_info = json.loads(payload)
        else:
            print(line, end="", flush=True)
    return_code = proc.wait()
bar.n = 100.0
bar.close()
plt.close(live_fig)
assert (
    return_code == 0
), f"Training failed with exit code {return_code}; see logs/train_fg.log"
print(pd.Series(done_info).to_string())

## 14. Training curves

Reloads `metrics/train_history.json`, saves it as CSV and draws the final curves: loss, validation PSNR against the frame-average and copy baselines, learning rate, throughput, and per-GPU memory and utilisation. The table shows the last rounds (with SSIM and interpolation error) and the best round. Expected output: falling loss and validation PSNR rising clearly above the frame average.

In [ ]:
history_df = pd.DataFrame(
    json.loads((DIRS["metrics"] / "train_history.json").read_text())
)
history_df["val_gain_over_avg_db"] = history_df["val_psnr"] - history_df["val_avg_psnr"]
history_df.to_csv(DIRS["metrics"] / "train_history.csv", index=False)
cols = [
    "round",
    "elapsed_min",
    "step",
    "train_loss",
    "train_psnr",
    "val_psnr",
    "val_ssim",
    "val_ie",
    "val_gain_over_avg_db",
    "lr",
    "triplets_per_s",
]
print(history_df[cols].tail(8).round(4).to_string(index=False))
best_round = history_df.loc[history_df["val_psnr"].idxmax()]
print(
    f"best round {int(best_round['round'])}: val PSNR {best_round['val_psnr']:.3f} dB, gain over frame average {best_round['val_gain_over_avg_db']:.3f} dB"
)
fig, axes = plt.subplots(2, 3, figsize=(16, 7.5))
draw_history(axes, history_df.to_dict("records"))
fig.suptitle("NeuralFG training (DDP, 2 x T4)")
show(fig, "training_curves")

## 15. Best weights and GPU replicas

Loads the best EMA weights, saves them as `weights/fg_model.pt`, and places one FP16 replica on each GPU (warping stays FP32 inside the model). Defines the helpers used by the evaluation cells: `run_on_gpus` splits work across both GPUs with one thread per replica, and `interpolate` returns the model output, the frame average and the auxiliary flow and blend maps. Expected output: the best round and parameter count.

In [ ]:
best_ckpt = torch.load(DIRS["weights"] / "fg_best_ema.pt", map_location="cpu")
model = NeuralFG(mid=CFG["mid_channels"], downsample=CFG["downsample"])
model.load_state_dict(best_ckpt["ema"])
model.eval()
torch.save(
    {
        "state_dict": model.state_dict(),
        "mid_channels": CFG["mid_channels"],
        "downsample": CFG["downsample"],
    },
    DIRS["weights"] / "fg_model.pt",
)
REPLICAS = [copy.deepcopy(model).half().to(f"cuda:{i}").eval() for i in range(N_GPUS)]
print(
    pd.Series(
        {
            "best_round": best_ckpt["round"],
            "best_val_psnr": round(best_ckpt["val_psnr"], 3),
            "params": count_params(model),
        }
    ).to_string()
)


def run_on_gpus(items, fn, desc):
    """Split items across GPUs (one thread per GPU replica) and return results in input order."""
    results = [None] * len(items)
    bar = tqdm(total=len(items), desc=desc)
    lock = threading.Lock()

    def worker(rank):
        device = torch.device("cuda", rank)
        for i in range(rank, len(items), N_GPUS):
            results[i] = fn(REPLICAS[rank], device, items[i])
            with lock:
                bar.update(1)

    with ThreadPoolExecutor(max_workers=N_GPUS) as pool:
        list(pool.map(worker, range(N_GPUS)))
    bar.close()
    return results


def to_tensor(img, device):
    return torch.from_numpy(img).to(device).permute(2, 0, 1)[None].float().div_(255.0)


@torch.no_grad()
def interpolate(net, device, paths):
    f0, gt, f1 = (to_tensor(load_rgb(p), device) for p in paths)
    pred, aux = net(f0.half(), f1.half())
    return f0, gt, f1, quantize(pred.float()), quantize((f0 + f1) / 2), aux


def triplet_metrics(f0, gt, f1, pred, avg):
    return {
        "psnr": psnr(pred, gt).item(),
        "ssim": ssim(pred, gt).item(),
        "ie": interpolation_error(pred, gt).item(),
        "avg_psnr": psnr(avg, gt).item(),
        "avg_ssim": ssim(avg, gt).item(),
        "avg_ie": interpolation_error(avg, gt).item(),
        "copy_psnr": psnr(f0, gt).item(),
        "motion": ((f0 - f1).abs().mean() * 255).item(),
    }

## 16. Test evaluation (Vimeo-90K test and held-out sequences)

Evaluates full frames, split across both GPUs: up to `test_max` Vimeo-90K test triplets (official list when present) and up to `holdout_full_per_dataset` held-out validation triplets per training dataset at native resolution (REDS 1280x720, Sintel 1024x436). For each triplet: NeuralFG, frame average and copy of t-1, scored with PSNR, SSIM and interpolation error, plus the motion size. Expected output: a per-set table (means, gain over the average, share of triplets improved) and `metrics/test_per_triplet.csv`, `metrics/test_summary.csv`.

In [ ]:
test_rows = catalogue[(catalogue["split"] == "test") & (catalogue["role"] == "train")]
test_rows = test_rows.sample(
    n=min(CFG["test_max"], len(test_rows)), random_state=CFG["seed"]
).assign(set="vimeo_test")
holdout_rows = (
    catalogue[catalogue["split"] == "val"]
    .groupby("dataset", group_keys=False)
    .apply(
        lambda d: d.sample(
            min(len(d), CFG["holdout_full_per_dataset"]), random_state=CFG["seed"]
        )
    )
)
holdout_rows = holdout_rows.assign(set="holdout_" + holdout_rows["dataset"])
eval_items = pd.concat([test_rows, holdout_rows])[["set", "f0", "gt", "f1"]].to_dict(
    "records"
)


def eval_triplet(net, device, item):
    try:
        f0, gt, f1, pred, avg, _ = interpolate(
            net, device, (item["f0"], item["gt"], item["f1"])
        )
    except Exception:
        return None
    if f0.shape != gt.shape or f0.shape != f1.shape:
        return None
    return {
        "set": item["set"],
        "triplet": str(Path(item["gt"]).parent),
        "gt_path": item["gt"],
        **triplet_metrics(f0, gt, f1, pred, avg),
    }


with stage("test_eval"):
    test_df = pd.DataFrame(
        [
            r
            for r in run_on_gpus(
                eval_items, eval_triplet, "Evaluating triplets (2 GPUs)"
            )
            if r is not None
        ]
    )
test_df["gain_over_avg"] = test_df["psnr"] - test_df["avg_psnr"]
test_summary = (
    test_df.groupby("set")
    .agg(
        triplets=("psnr", "size"),
        psnr=("psnr", "mean"),
        ssim=("ssim", "mean"),
        ie=("ie", "mean"),
        avg_psnr=("avg_psnr", "mean"),
        avg_ssim=("avg_ssim", "mean"),
        copy_psnr=("copy_psnr", "mean"),
        gain_over_avg=("gain_over_avg", "mean"),
        improved_pct=("gain_over_avg", lambda g: 100 * (g > 0).mean()),
        motion=("motion", "mean"),
    )
    .round(4)
)
print(test_summary.to_string())
test_df.to_csv(DIRS["metrics"] / "test_per_triplet.csv", index=False)
test_summary.to_csv(DIRS["metrics"] / "test_summary.csv")

## 17. Result plots and error analysis by motion

Bar chart of NeuralFG, frame average and copy PSNR per set; the gain over the average as a function of motion size (binned by quantile), which shows where the learned flow helps and where large motion still defeats the quarter-resolution flow; and the triplets with the largest and smallest gains. Expected output: gains growing with moderate motion and shrinking or turning negative on the largest motions.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
x = np.arange(len(test_summary))
for offset, column, label, color in (
    (-0.27, "copy_psnr", "copy t-1", "#8C8C8C"),
    (0.0, "avg_psnr", "frame average", "#DD8452"),
    (0.27, "psnr", "NeuralFG", "#4C72B0"),
):
    axes[0].bar(x + offset, test_summary[column], 0.27, label=label, color=color)
axes[0].set_xticks(x, test_summary.index, rotation=20)
axes[0].set_ylim(test_summary["copy_psnr"].min() - 2, test_summary["psnr"].max() + 1.5)
axes[0].set_ylabel("PSNR (dB)")
axes[0].legend()
axes[0].set_title("PSNR by test set")
test_df["motion_bin"] = pd.qcut(test_df["motion"], q=8, duplicates="drop")
by_motion = test_df.groupby("motion_bin", observed=True).agg(
    gain=("gain_over_avg", "mean"), psnr=("psnr", "mean"), avg_psnr=("avg_psnr", "mean")
)
centres = [interval.mid for interval in by_motion.index]
axes[1].plot(centres, by_motion["psnr"], "o-", label="NeuralFG")
axes[1].plot(centres, by_motion["avg_psnr"], "o-", label="frame average")
axes[1].set_xlabel("motion (mean |t-1 - t|, 0-255)")
axes[1].set_ylabel("PSNR (dB)")
axes[1].legend()
axes[1].set_title("PSNR against motion size")
axes[2].scatter(test_df["motion"], test_df["gain_over_avg"], s=6, alpha=0.4)
axes[2].axhline(0, ls="--", color="#C44E52")
axes[2].set_xlabel("motion (0-255)")
axes[2].set_ylabel("gain over frame average (dB)")
axes[2].set_title("Per-triplet gain")
show(fig, "test_results")
by_motion.to_csv(DIRS["metrics"] / "gain_by_motion.csv")
show_cols = ["set", "triplet", "psnr", "avg_psnr", "gain_over_avg", "motion"]
print(
    "largest gains:\n"
    + test_df.nlargest(5, "gain_over_avg")[show_cols].round(3).to_string(index=False)
)
print(
    "smallest gains:\n"
    + test_df.nsmallest(5, "gain_over_avg")[show_cols].round(3).to_string(index=False)
)

## 18. Visual comparisons, flow and blend weights

For four test triplets spread over the motion range: the two inputs, the frame average, NeuralFG, the ground truth, NeuralFG's absolute error, the predicted flow from t-0.5 to t-1 (colour wheel: hue is direction, brightness is magnitude) and the blend weight given to frame t-1 (bright where t-1 is trusted, dark where t is). Interpolated frames are saved to `predictions/`. Expected output: sharper moving edges than the average, with errors concentrated at occlusion boundaries.

In [ ]:
def flow_to_rgb(flow):
    fx, fy = flow[0], flow[1]
    magnitude = np.sqrt(fx**2 + fy**2)
    hsv = np.stack(
        [
            (np.arctan2(fy, fx) + np.pi) / (2 * np.pi),
            np.ones_like(fx),
            np.clip(magnitude / max(np.percentile(magnitude, 99), 1e-6), 0, 1),
        ],
        axis=-1,
    )
    return hsv_to_rgb(hsv)


def to_np(t):
    return t[0].permute(1, 2, 0).float().cpu().numpy()


vis = test_df[test_df["set"] == test_df["set"].iloc[0]].sort_values("motion")
vis = vis.iloc[np.linspace(0, len(vis) - 1, min(4, len(vis))).astype(int)]
titles = [
    "t-1",
    "t",
    "frame average",
    "NeuralFG",
    "GT",
    "|NeuralFG - GT|",
    "flow to t-1",
    "weight of t-1",
]
fig, axes = plt.subplots(len(vis), 8, figsize=(24, 2.9 * len(vis)), squeeze=False)
for row_axes, r in zip(
    axes, tqdm(list(vis.itertuples()), desc="Rendering comparisons")
):
    folder = Path(r.gt_path).parent
    item = catalogue[catalogue["gt"] == r.gt_path].iloc[0]
    f0, gt, f1, pred, avg, aux = interpolate(
        REPLICAS[0], torch.device("cuda", 0), (item["f0"], item["gt"], item["f1"])
    )
    panels = [
        to_np(f0),
        to_np(f1),
        to_np(avg),
        to_np(pred),
        to_np(gt),
        np.abs(to_np(pred) - to_np(gt)).mean(axis=2),
        flow_to_rgb(aux["flow0"][0].float().cpu().numpy()),
        aux["weight0"][0, 0].float().cpu().numpy(),
    ]
    for ax, img, title in zip(row_axes, panels, titles):
        ax.imshow(
            np.clip(img, 0, 1) if img.ndim == 3 else img,
            cmap=None if img.ndim == 3 else "magma",
            vmin=None if img.ndim == 3 else 0,
            vmax=None if img.ndim == 3 else (0.15 if title.startswith("|") else 1),
        )
        ax.set_title(
            f"{title}" + (f" ({r.psnr:.2f} dB)" if title == "NeuralFG" else ""),
            fontsize=8,
        )
        ax.axis("off")
    Image.fromarray((to_np(pred) * 255).round().astype(np.uint8)).save(
        DIRS["predictions"] / f"{folder.parent.name}_{folder.name}_mid.png"
    )
fig.suptitle(f"NeuralFG on {vis['set'].iloc[0]}, from least to most motion")
show(fig, "visual_comparisons")

## 19. Vid4 zero-shot evaluation

Builds every consecutive triplet inside each Vid4 sequence (frames i-1 and i+1 in, frame i as ground truth), splits sequences across both GPUs, and reports per-sequence PSNR, SSIM and interpolation error for NeuralFG and the frame average. Vid4 was never used for training or model selection. Expected output: a per-sequence table, per-frame PSNR curves and `metrics/vid4_*.csv`.

In [ ]:
vid4_cat = catalogue[catalogue["role"] == "video_test"].copy()
vid4_cat["sequence"] = vid4_cat["gt"].map(sequence_name)
sequences = [
    {"sequence": s, "triplets": list(zip(g["f0"], g["gt"], g["f1"]))}
    for s, g in vid4_cat.groupby("sequence")
]


def eval_sequence(net, device, item):
    rows = []
    for idx, paths in enumerate(item["triplets"]):
        f0, gt, f1, pred, avg, _ = interpolate(net, device, paths)
        rows.append(
            {
                "sequence": item["sequence"],
                "frame": idx,
                **triplet_metrics(f0, gt, f1, pred, avg),
            }
        )
    return rows


if sequences:
    with stage("vid4_eval"):
        vid4_df = pd.DataFrame(
            [
                r
                for rows in run_on_gpus(
                    sequences, eval_sequence, "Vid4 sequences (2 GPUs)"
                )
                for r in rows
            ]
        )
    vid4_summary = vid4_df.groupby("sequence").agg(
        triplets=("psnr", "size"),
        psnr=("psnr", "mean"),
        ssim=("ssim", "mean"),
        ie=("ie", "mean"),
        avg_psnr=("avg_psnr", "mean"),
        motion=("motion", "mean"),
    )
    vid4_summary.loc["mean"] = vid4_summary.mean(numeric_only=True)
    print(vid4_summary.round(4).to_string())
    vid4_df.to_csv(DIRS["metrics"] / "vid4_per_frame.csv", index=False)
    vid4_summary.to_csv(DIRS["metrics"] / "vid4_summary.csv")
    fig, ax = plt.subplots(figsize=(12, 4.2))
    for name, group in vid4_df.groupby("sequence"):
        (line,) = ax.plot(group["frame"], group["psnr"], label=f"{name} NeuralFG")
        ax.plot(
            group["frame"],
            group["avg_psnr"],
            ls="--",
            color=line.get_color(),
            alpha=0.6,
        )
    ax.set_xlabel("triplet index")
    ax.set_ylabel("PSNR (dB)")
    ax.set_title("Vid4 zero-shot interpolation (dashed: frame average)")
    ax.legend(fontsize=8)
    show(fig, "vid4_results")
else:
    vid4_summary = None
    log.warning("Vid4 not attached; zero-shot evaluation skipped")

## 20. Inference latency

Times the FP16 model on GPU 0 with CUDA events (warm-up, then `latency_iters` timed runs) for two 960x540 frames and two 1920x1080 frames. This is a T4 measurement through PyTorch, not the DirectML or HLSL deployment path, so it ranks variants rather than predicting in-engine latency. Expected output: mean, P50 and P95 latency, frames per second, peak memory and a histogram.

In [ ]:
@torch.no_grad()
def time_model(net, size, iters):
    device = torch.device("cuda", 0)
    f0, f1 = (torch.rand(1, 3, *size, device=device).half() for _ in range(2))
    for _ in range(10):
        net(f0, f1)
    torch.cuda.synchronize(device)
    torch.cuda.reset_peak_memory_stats(device)
    times = []
    for _ in tqdm(range(iters), desc=f"Timing {size[1]}x{size[0]}", leave=False):
        start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(
            enable_timing=True
        )
        start.record()
        net(f0, f1)
        end.record()
        torch.cuda.synchronize(device)
        times.append(start.elapsed_time(end))
    return np.array(times), torch.cuda.max_memory_allocated(device) / 2**20


with stage("latency"):
    latency_rows, latency_samples = [], {}
    for h, w in CFG["latency_sizes"]:
        times, peak_mib = time_model(REPLICAS[0], (h, w), CFG["latency_iters"])
        name = f"{w}x{h}"
        latency_samples[name] = times
        latency_rows.append(
            {
                "input": name,
                "mean_ms": times.mean(),
                "p50_ms": np.median(times),
                "p95_ms": np.percentile(times, 95),
                "fps": 1000 / times.mean(),
                "peak_mem_mib": peak_mib,
            }
        )
latency_df = pd.DataFrame(latency_rows).round(3)
print(latency_df.to_string(index=False))
latency_df.to_csv(DIRS["metrics"] / "latency.csv", index=False)
fig, ax = plt.subplots(figsize=(9, 4))
for name, times in latency_samples.items():
    ax.hist(times, bins=40, alpha=0.6, label=name)
ax.set_xlabel("GPU time per interpolated frame (ms, T4, FP16 CNN, FP32 warp)")
ax.set_ylabel("runs")
ax.legend()
ax.set_title("NeuralFG inference latency")
show(fig, "latency")

## 21. ONNX export

Exports the FP32 model to `weights/neural_fg.onnx` (opset 17, which includes `GridSample`; dynamic height and width), trying the TorchScript exporter first and the default exporter second, then checks it with ONNX Runtime when available. The exported graph is traced at 64x64, so the replicate-padding branch is not included: feed sizes divisible by 4 (960x540 and 1920x1080 are). Failures are reported, not raised. Expected output: a status table.

In [ ]:
with stage("onnx_export"):
    onnx_path = DIRS["weights"] / "neural_fg.onnx"
    export_model = FGInference(copy.deepcopy(model).float().cpu().eval()).eval()
    dummy = (torch.rand(1, 3, 64, 64), torch.rand(1, 3, 64, 64))
    export_status = {"exported": False, "error": None, "onnxruntime_max_abs_diff": None}
    for kwargs in ({"dynamo": False}, {}):
        try:
            torch.onnx.export(
                export_model,
                dummy,
                str(onnx_path),
                input_names=["frame_prev", "frame_curr"],
                output_names=["frame_mid"],
                opset_version=17,
                dynamic_axes={
                    n: {2: "h", 3: "w"}
                    for n in ("frame_prev", "frame_curr", "frame_mid")
                },
                **kwargs,
            )
            export_status.update(exported=True, error=None)
            break
        except Exception as exc:
            export_status["error"] = repr(exc)[:300]
    if export_status["exported"]:
        try:
            import onnxruntime as ort

            session = ort.InferenceSession(
                str(onnx_path), providers=["CPUExecutionProvider"]
            )
            t0, t1 = torch.rand(1, 3, 96, 160), torch.rand(1, 3, 96, 160)
            onnx_out = session.run(
                None, {"frame_prev": t0.numpy(), "frame_curr": t1.numpy()}
            )[0]
            with torch.no_grad():
                export_status["onnxruntime_max_abs_diff"] = float(
                    np.abs(onnx_out - export_model(t0, t1).numpy()).max()
                )
        except Exception as exc:
            export_status["onnxruntime_check"] = f"skipped: {exc!r}"[:200]
        export_status["onnx_mb"] = round(onnx_path.stat().st_size / 2**20, 3)
print(pd.Series(export_status).to_string())
save_json(export_status, "onnx_export.json")

## 22. Summary

Collects the key results into one table (`metrics/summary.json`) and reports the run time of every stage against the 8 hour budget (`metrics/stage_times.csv`, bar chart). Expected output: the summary, the stage table and a total below 8 hours.

In [ ]:
summary = {
    "model": "NeuralFG (RGB, 6-channel input)",
    "params": count_params(model),
    "train_minutes": round(STAGE_TIMES.get("training", 0) / 60, 1),
    "train_steps": done_info.get("steps"),
    "batch_per_gpu": BATCH_PER_GPU,
    "effective_batch": EFFECTIVE_BATCH,
    "mean_throughput_triplets_s": round(history_df["triplets_per_s"].mean(), 1),
    "peak_mem_gb_per_gpu": history_df["mem_gb"].iloc[-1],
    "best_val_psnr": round(best_round["val_psnr"], 3),
    "best_val_gain_over_avg_db": round(best_round["val_gain_over_avg_db"], 3),
}
for name, row in test_summary.iterrows():
    summary[f"{name}_psnr"] = row["psnr"]
    summary[f"{name}_ssim"] = row["ssim"]
    summary[f"{name}_gain_over_avg_db"] = row["gain_over_avg"]
if vid4_summary is not None:
    summary["vid4_psnr"] = round(vid4_summary.loc["mean", "psnr"], 3)
    summary["vid4_avg_psnr"] = round(vid4_summary.loc["mean", "avg_psnr"], 3)
for _, row in latency_df.iterrows():
    summary[f"latency_p95_ms_{row['input']}"] = row["p95_ms"]
summary["onnx_exported"] = export_status["exported"]
summary["run_hours_so_far"] = round((time.time() - NOTEBOOK_START) / 3600, 2)
print(pd.Series(summary).to_string())
save_json(summary, "summary.json")

stage_df = pd.Series(STAGE_TIMES, name="seconds").to_frame()
stage_df["minutes"] = (stage_df["seconds"] / 60).round(1)
stage_df.to_csv(DIRS["metrics"] / "stage_times.csv")
print(stage_df.to_string())
fig, ax = plt.subplots(figsize=(9, 3.8))
stage_df["minutes"].plot.barh(ax=ax, color="#4C72B0")
ax.set_xlabel("minutes")
ax.set_title(
    f"Stage run times (total so far {summary['run_hours_so_far']} h of {RUN_BUDGET_H} h)"
)
show(fig, "stage_times")

## 23. Remove the temporary triplet cache

The cache is an intermediate, not an output: deleting it keeps the saved Kaggle output small and frees disk for the zip. Expected output: the space freed.

In [ ]:
cache_bytes = sum(p.stat().st_size for p in DIRS["cache"].rglob("*") if p.is_file())
shutil.rmtree(DIRS["cache"], ignore_errors=True)
print(f"removed cache: {cache_bytes / 1e9:.2f} GB freed")

## 24. Package outputs

Zips everything in `/kaggle/working` (weights, plots, metrics, logs, predictions, code) into `outputs.zip`, storing already-compressed files without recompression, after checking there is enough disk. The link below downloads it; after a **Save & Run All** version, the zip and all files also appear in the notebook's **Output** tab.

In [ ]:
with stage("packaging"):
    ZIP_PATH = WORK / "outputs.zip"
    EXCLUDE_PARTS = {
        "__pycache__",
        ".cache",
        ".ipynb_checkpoints",
        "cache",
        "tmp",
        "wandb",
    }
    STORED_SUFFIXES = {
        ".zip",
        ".png",
        ".jpg",
        ".jpeg",
        ".pt",
        ".pth",
        ".bin",
        ".safetensors",
        ".gz",
        ".npz",
        ".onnx",
    }
    files = [
        path
        for path in WORK.rglob("*")
        if path.is_file()
        and path != ZIP_PATH
        and not EXCLUDE_PARTS.intersection(path.relative_to(WORK).parts)
    ]
    total_bytes = sum(path.stat().st_size for path in files)
    free_bytes = shutil.disk_usage(WORK).free
    assert (
        free_bytes > total_bytes * 1.05
    ), f"Not enough disk space to zip {total_bytes / 1e9:.2f} GB with {free_bytes / 1e9:.2f} GB free."
    with zipfile.ZipFile(ZIP_PATH, "w", allowZip64=True) as archive:
        for path in tqdm(files, desc="Zipping outputs", unit="file"):
            method = (
                zipfile.ZIP_STORED
                if path.suffix.lower() in STORED_SUFFIXES
                else zipfile.ZIP_DEFLATED
            )
            archive.write(path, path.relative_to(WORK), compress_type=method)
print(
    f"Zipped {len(files)} files ({total_bytes / 1e9:.2f} GB) into {ZIP_PATH}; total run time {(time.time() - NOTEBOOK_START) / 3600:.2f} h",
    flush=True,
)
display(FileLink(os.path.relpath(ZIP_PATH)))